In [1]:
pip install tensorflow opencv-python numpy pandas

Note: you may need to restart the kernel to use updated packages.


In [2]:
pip install deepface

Note: you may need to restart the kernel to use updated packages.


In [3]:
pip install deepface opencv-python

Note: you may need to restart the kernel to use updated packages.


In [4]:
from deepface import DeepFace
import cv2
import numpy as np
import csv
import os
from datetime import datetime

In [26]:
def load_students(folder='Students_Images'):
    students = {}

    for filename in os.listdir(folder):
        if filename.lower().endswith(('.jpg', '.jpeg', '.png')):
            name = os.path.splitext(filename)[0]
            path = os.path.join(folder, filename)
            students[name] = path

    return students

students = load_students()
print(students)

{'Alexandra Daddario_45': 'Students_Images\\Alexandra Daddario_45.jpg', 'Alexandra Daddario_48': 'Students_Images\\Alexandra Daddario_48.jpg', 'Alexandra Daddario_89': 'Students_Images\\Alexandra Daddario_89.jpg', 'Alia Bhatt_13': 'Students_Images\\Alia Bhatt_13.jpg', 'Billie Eilish_5': 'Students_Images\\Billie Eilish_5.jpg', 'Brad Pitt_3': 'Students_Images\\Brad Pitt_3.jpg', 'Camila Cabello_6': 'Students_Images\\Camila Cabello_6.jpg', 'Charlize Theron_1': 'Students_Images\\Charlize Theron_1.jpg', 'Henry Cavill_4': 'Students_Images\\Henry Cavill_4.jpg', 'Henry Cavill_6': 'Students_Images\\Henry Cavill_6.jpg', 'Hrithik Roshan_1': 'Students_Images\\Hrithik Roshan_1.jpg', 'Hugh Jackman_2': 'Students_Images\\Hugh Jackman_2.jpg', 'Kashyap_15': 'Students_Images\\Kashyap_15.jpg', 'Lisa Kudrow_1': 'Students_Images\\Lisa Kudrow_1.jpg', 'malak.jpg': 'Students_Images\\malak.jpg.jpg'}


In [28]:
def mark_attendance(name, status, csv_file='attendance.csv'):
    now      = datetime.now()
    date_str = now.strftime('%Y-%m-%d')
    time_str = now.strftime('%H:%M:%S') if status != 'Absent' else '---'

    if os.path.exists(csv_file):
        with open(csv_file, 'r', encoding='utf-8-sig') as f:
            reader = csv.DictReader(f)
            for row in reader:
                if row.get('Name') == name and row.get('Date') == date_str:
                    return False

    file_exists = os.path.exists(csv_file)
    with open(csv_file, 'a', newline='', encoding='utf-8-sig') as f:
        fieldnames = ['Name', 'Date', 'Arrival_Time', 'Status']
        writer = csv.DictWriter(f, fieldnames=fieldnames)
        if not file_exists:
            writer.writeheader()
        writer.writerow({
            'Name'        : name,
            'Date'        : date_str,
            'Arrival_Time': time_str,
            'Status'      : status
        })
    return True

In [30]:
def identify_face(frame, students):
    temp_path = 'temp_frame.jpg'
    cv2.imwrite(temp_path, frame)

    for name, student_img_path in students.items():
        try:
            result = DeepFace.verify(
                img1_path         = temp_path,
                img2_path         = student_img_path,
                enforce_detection = False,
                model_name        = 'Facenet'
            )
            if result['verified']:
                os.remove(temp_path)
                return name
        except Exception:
            continue

    if os.path.exists(temp_path):
        os.remove(temp_path)
    return None

In [31]:
LATE_HOUR   = 8
LATE_MINUTE = 15

students = load_students('Students_Images')


cap = cv2.VideoCapture(0)
count = 0
logged = {}

while True:
    success, frame = cap.read()
    if not success:
        break

    count += 1

   
    if count % 30 == 0:
        name = identify_face(frame, students)

        if name and name not in logged:
            now = datetime.now()
            if now.hour < LATE_HOUR or (now.hour == LATE_HOUR and now.minute < LATE_MINUTE):
                status = 'Present'
            else:
                status = 'Late'

            mark_attendance(name, status)
            logged[name] = status
            print(name)
            print(status)

   
    now_str = datetime.now().strftime('%H:%M:%S')
    cv2.putText(frame, f'Time: {now_str}', (10, 30),
                cv2.FONT_HERSHEY_SIMPLEX, 0.7, (255, 255, 255), 2)
    cv2.putText(frame, f'Present: {len(logged)}/{len(students)}', (10, 60),
                cv2.FONT_HERSHEY_SIMPLEX, 0.7, (0, 255, 0), 2)

    y = 100
    for n, s in logged.items():
        color = (0, 255, 0) if s == 'Present' else (0, 165, 255)
        cv2.putText(frame, f'{n}: {s}', (10, y),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.6, color, 2)
        y += 25

    cv2.imshow('Attendance System', frame)

   
    if cv2.waitKey(1) & 0xFF == ord('q'):
        break


cap.release()
cv2.destroyAllWindows()

Alexandra Daddario_48
Late
Alia Bhatt_13
Late


In [29]:
for name in students:
    if name in logged:
        print(name, '|', logged[name])
    else:
        print(name, '| Absent')
        mark_attendance(name, 'Absent')

print('Present :', sum(1 for v in logged.values() if v == 'Present'))
print('Late    :', sum(1 for v in logged.values() if v == 'Late'))
print('Absent  :', len([n for n in students if n not in logged]))

Alexandra Daddario_45 | Absent
Alexandra Daddario_48 | Absent
Alexandra Daddario_89 | Absent
Alia Bhatt_13 | Absent
Billie Eilish_5 | Absent
Brad Pitt_3 | Absent
Camila Cabello_6 | Absent
Charlize Theron_1 | Absent
Henry Cavill_4 | Absent
Henry Cavill_6 | Absent
Hrithik Roshan_1 | Absent
Hugh Jackman_2 | Absent
Kashyap_15 | Absent
Lisa Kudrow_1 | Absent
malak.jpg | Late
Present : 0
Late    : 1
Absent  : 14
